In [3]:
import kagglehub

kagglehub.login()

In [4]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
import zipfile
zip=zipfile.ZipFile('utkface-new.zip')
zip.extractall()
zip.close()

In [6]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
folder='/content/utkface_aligned_cropped/UTKFace'

In [8]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [9]:
len(age)

23708

In [10]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [11]:
df.shape

(23708, 3)

In [12]:
df.head()

,age,gender,img
0,31,0,31_0_2_20170116193441821.jpg.chip.jpg
1,28,0,28_0_0_20170117133953048.jpg.chip.jpg
2,54,0,54_0_1_20170113135022114.jpg.chip.jpg
3,58,0,58_0_3_20170119211548329.jpg.chip.jpg
4,26,1,26_1_3_20170119193139994.jpg.chip.jpg


In [13]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [14]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [15]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [17]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [18]:
resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [19]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [20]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [24]:
print(type(train_generator))
print(type(test_generator))

<class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>
<class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>


In [28]:
x, y = next(train_generator)

print(type(x))
print(type(y))
print(y)

<class 'numpy.ndarray'>
<class 'list'>
[array([103,  26,   2,  32,   3,  56,  53,  51,  54,  73,  54,  25,  38,
        26,   1,  38,  26,  72,   2,  55,  68,  26,  26,  23,  28,  26,
        22,  45,  20,  28,  40,  36]), array([0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1])]


In [ ]:
import tensorflow as tf

# Define the generator_wrapper function to convert list outputs to dictionary outputs
def generator_wrapper(generator):
    for x_batch, y_list_batch in generator:
        # y_list_batch[0] contains age labels
        # y_list_batch[1] contains gender labels
        # Cast to float32 as typically expected by Keras loss functions (mae, binary_crossentropy)
        yield x_batch, {
            'age': tf.cast(y_list_batch[0], tf.float32),
            'gender': tf.cast(y_list_batch[1], tf.float32)
        }

# Recreate train_dataset and test_dataset with the correct generator_wrapper
# and updated output_signature to match the dictionary output for labels
train_dataset = tf.data.Dataset.from_generator(
    lambda: generator_wrapper(train_generator),
    output_signature=(
        tf.TensorSpec(shape=(None, 200, 200, 3), dtype=tf.float32),
        {
            'age': tf.TensorSpec(shape=(None,), dtype=tf.float32),  # Age for regression
            'gender': tf.TensorSpec(shape=(None,), dtype=tf.float32) # Gender for binary classification
        }
    )
)

test_dataset = tf.data.Dataset.from_generator(
    lambda: generator_wrapper(test_generator),
    output_signature=(
        tf.TensorSpec(shape=(None, 200, 200, 3), dtype=tf.float32),
        {
            'age': tf.TensorSpec(shape=(None,), dtype=tf.float32),
            'gender': tf.TensorSpec(shape=(None,), dtype=tf.float32)
        }
    )
)

# Calculate steps per epoch and validation steps based on generator's total samples and batch size
batch_size = train_generator.batch_size # Assuming batch_size is consistent for both generators
num_train_samples = train_generator.n
num_test_samples = test_generator.n

steps_per_epoch = num_train_samples // batch_size
if num_train_samples % batch_size != 0:
    steps_per_epoch += 1

validation_steps = num_test_samples // batch_size
if num_test_samples % batch_size != 0:
    validation_steps += 1

history = model.fit(
    train_dataset,
    epochs=10,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_dataset,
    validation_steps=validation_steps
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 266s 407ms/step - age_loss: 15.5127 - age_mae: 15.5127 - gender_accuracy: 0.5117 - gender_loss: 0.9024 - loss: 104.8496 - val_age_loss: 14.4562 - val_age_mae: 14.4567 - val_gender_accuracy: 0.5213 - val_gender_loss: 0.6922 - val_loss: 82.9884
Epoch 2/10
350/625 ━━━━━━━━━━━━━━━━━━━━ 1:39 361ms/step - age_loss: 14.7497 - age_mae: 14.7497 - gender_accuracy: 0.5295 - gender_loss: 0.6937 - loss: 83.4282

In [ ]:
results = model.evaluate(test_generator, return_dict=True)

print(results)